# PRW Person Search Training and Evaluation

This notebook trains on a deterministic 90% subset of the official PRW training frames, validates on the remaining 10%, saves resumable checkpoints, and evaluates once on the untouched official PRW test set. The model uses contrastive ReID loss and Faster R-CNN-style matching: positive IoU >= 0.7, negative IoU < 0.3, and the interval between them is ignored.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

PROJECT_INPUT = Path('/kaggle/input/datasets/saadqasimabbasi/ml4cv-ass')
PROJECT_WORKING = Path('/kaggle/working/ml4cv-ass')
PRW_ROOT = Path('/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild')

# Refresh code from the attached project dataset without touching checkpoints.
shutil.copytree(PROJECT_INPUT, PROJECT_WORKING, dirs_exist_ok=True)
SRC_DIR = PROJECT_WORKING / 'src'
sys.path.insert(0, str(SRC_DIR))
%cd /kaggle/working/ml4cv-ass/src

In [ ]:
import random
import torch
import wandb
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader

from model import PersonSearchModel
from prw_dataset import (PRWPersonSearchDataset, PRWSubset,
                         PKIdentityBatchSampler, prw_collate_fn)
from training import make_grad_scaler, start_run, train_one_epoch, validate

wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB')

In [ ]:
IMAGE_SIZE = (256, 384)
VAL_FRACTION = 0.10
P, K = 4, 2
NUM_WORKERS = 2

full_train = PRWPersonSearchDataset(PRW_ROOT, split='train', image_size=IMAGE_SIZE)
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(full_train), generator=generator).tolist()
val_count = round(len(indices) * VAL_FRACTION)
val_dataset = PRWSubset(full_train, indices[:val_count])
train_dataset = PRWSubset(full_train, indices[val_count:])
train_sampler = PKIdentityBatchSampler(train_dataset, p=P, k=K)

train_loader = DataLoader(
    train_dataset, batch_sampler=train_sampler, collate_fn=prw_collate_fn,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset, batch_size=2, shuffle=False, collate_fn=prw_collate_fn,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
print(f'train frames: {len(train_dataset)}, validation frames: {len(val_dataset)}')
print(f'PK batches per epoch: {len(train_sampler)}, nominal batch size: {P*K}')

## Train, Validate, and Checkpoint

Run this cell for the complete training job. It validates every epoch, saves numbered and latest checkpoints every epoch, keeps the checkpoint with the lowest validation loss, uploads that best checkpoint to W&B, and restores it before test evaluation.

In [ ]:
NUM_EPOCHS = 20
VAL_MAX_BATCHES = 20
CHECKPOINT_EVERY = 1
WANDB_CHECKPOINT_EVERY = 5  # persist latest.pt outside the Kaggle session
TRAIN_LR = 1e-4
WEIGHT_DECAY = 1e-4
ADAMW_BETAS = (0.9, 0.999)
ADAMW_EPS = 1e-8
RESUME_FROM = None  # Example: PROJECT_WORKING / 'checkpoints/latest.pt'

model_config = dict(
    backbone_name='resnet50', pretrained=True, metric_loss='contrastive',
    anchor_sizes=((16,), (32,), (64,), (128,)),
    anchor_ratios=((2.0, 3.0, 4.0),) * 4,
    rpn_fg_iou_thresh=0.7, rpn_bg_iou_thresh=0.3,
    roi_fg_iou_thresh=0.7, roi_bg_iou_thresh=0.3,
    rpn_pre_nms_top_n_train=1000, rpn_post_nms_top_n_train=512,
    rpn_pre_nms_top_n_test=500, rpn_post_nms_top_n_test=100,
)
run_config = {
    **model_config, 'dataset': 'PRW', 'train_split': 'official train (90%)',
    'validation_split': 'official train (10%)', 'test_split': 'official test',
    'image_size': IMAGE_SIZE, 'seed': SEED, 'P': P, 'K': K,
    'nominal_batch_size': P * K, 'epochs': NUM_EPOCHS,
    'optimizer': 'AdamW', 'lr': TRAIN_LR, 'weight_decay': WEIGHT_DECAY,
    'adamw_betas': ADAMW_BETAS, 'adamw_eps': ADAMW_EPS,
    'pretrained_weights': 'torchvision ResNet50 DEFAULT (ImageNet-1K)',
    'amp': device.type == 'cuda',
    'device': str(device),
    'gpu_name': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu',
}
run = start_run(run_config, name='prw-contrastive-adamw-imagenet', mode='online')
model = PersonSearchModel(**model_config).to(device)
optimizer = torch.optim.AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=TRAIN_LR, weight_decay=WEIGHT_DECAY, betas=ADAMW_BETAS, eps=ADAMW_EPS,
)
scaler = make_grad_scaler(device)
checkpoint_dir = PROJECT_WORKING / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
start_epoch, global_step, best_val_total = 0, 0, float('inf')

if RESUME_FROM is not None:
    checkpoint = torch.load(RESUME_FROM, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    checkpoint_optimizer = checkpoint.get('run_config', {}).get('optimizer')
    if checkpoint_optimizer == run_config['optimizer']:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    else:
        print(f'not restoring {checkpoint_optimizer or "unknown"} optimizer state; '
              f'starting a fresh {run_config["optimizer"]} optimizer')
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch']
    global_step = checkpoint['step']
    best_val_total = checkpoint.get('best_val_total', float('inf'))
    print(f'resuming after epoch {start_epoch} from {RESUME_FROM}')

for epoch in range(start_epoch, NUM_EPOCHS):
    history = train_one_epoch(
        model, train_loader, optimizer, device, max_steps=None, run=run,
        scaler=scaler, start_step=global_step,
    )
    global_step += len(history.get('total', []))
    val_metrics = validate(
        model, val_loader, device, run=run, step=max(global_step - 1, 0),
        max_batches=VAL_MAX_BATCHES,
    )
    improved = val_metrics['total'] < best_val_total
    if improved:
        best_val_total = val_metrics['total']
    checkpoint = {
        'epoch': epoch + 1, 'step': global_step, 'best_val_total': best_val_total,
        'model_config': model_config, 'run_config': run_config,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(), 'val_metrics': val_metrics,
    }
    torch.save(checkpoint, checkpoint_dir / 'latest.pt')
    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        torch.save(checkpoint, checkpoint_dir / f'epoch_{epoch + 1:03d}.pt')
    if improved:
        torch.save(checkpoint, checkpoint_dir / 'best.pt')
    if (epoch + 1) % WANDB_CHECKPOINT_EVERY == 0:
        checkpoint_artifact = wandb.Artifact(
            'prw-training-checkpoint', type='model',
            metadata={'epoch': epoch + 1, 'step': global_step,
                      'best_val_total': best_val_total},
        )
        checkpoint_artifact.add_file(str(checkpoint_dir / 'latest.pt'))
        run.log_artifact(checkpoint_artifact, aliases=['latest', f'epoch-{epoch + 1}'])
    run.summary['best/val_total_loss'] = best_val_total
    print(f"epoch {epoch + 1}/{NUM_EPOCHS}: val_total={val_metrics['total']:.4f}, "
          f"best={best_val_total:.4f}, checkpoint={checkpoint_dir / 'latest.pt'}")

best_checkpoint = torch.load(checkpoint_dir / 'best.pt', map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
best_artifact = wandb.Artifact('prw-best-model', type='model', metadata=run_config)
best_artifact.add_file(str(checkpoint_dir / 'best.pt'))
run.log_artifact(best_artifact, aliases=['best'])
print(f"training complete; restored best checkpoint from epoch {best_checkpoint['epoch']}")

## Final PRW Test Evaluation

Run this cell after training. It evaluates the restored best checkpoint on the official test gallery and queries, records mAP and top-1 in the W&B summary, and logs an interactive top-5 retrieval table. The table shows actual predicted person crops alongside gallery frames with box overlays.

In [ ]:
import importlib
import prw_evaluation
importlib.reload(prw_evaluation)
from prw_evaluation import evaluate_prw, make_wandb_retrieval_table

retrieval_metrics = evaluate_prw(
    model, PRW_ROOT, device, batch_size=2, score_threshold=0.0,
    iou_threshold=0.5, return_records=True, table_queries=100, table_top_k=5,
    query_batch_size=64, ranking_batch_size=8,
)
table_records = retrieval_metrics.pop('retrieval_records')
retrieval_table = make_wandb_retrieval_table(table_records, PRW_ROOT)
test_summary = {
    'test/mAP': retrieval_metrics['retrieval_mAP'],
    'test/top1': retrieval_metrics['retrieval_top1'],
    'test/evaluated_queries': retrieval_metrics['retrieval_queries'],
    'test/gallery_frames': retrieval_metrics['retrieval_gallery_frames'],
    'test/detections': retrieval_metrics['retrieval_detections'],
}
run.log(test_summary)
run.log({'test/query_top5_retrievals': retrieval_table})
for key, value in test_summary.items():
    run.summary[key] = value
print(f"FINAL PRW TEST: mAP={test_summary['test/mAP']:.4f}, "
      f"top-1={test_summary['test/top1']:.4f}")
run.finish()

Checkpoints are written under `/kaggle/working/ml4cv-ass/checkpoints`. Kaggle working storage is temporary, while the best checkpoint is also uploaded as the W&B model artifact `prw-best-model`.

## Inspect top predicted person boxes
Run this after evaluation. Change `QUERY_INDEX` to browse the saved queries; `TOP_K` is limited to `table_top_k` from evaluation. The first row shows the query crop and ranked predicted crops; the second shows each prediction in its full gallery frame. Similarity determines rank; green/red indicates whether the prediction matches the query identity under the evaluation rules.

In [ ]:
import matplotlib.pyplot as plt
from prw_evaluation import plot_retrieval_query

QUERY_INDEX = 0
TOP_K = 5

if table_records:
    print(f"Showing saved query {QUERY_INDEX} of {len(table_records)} (zero-based index)")
    fig = plot_retrieval_query(table_records, PRW_ROOT, query_index=QUERY_INDEX, top_k=TOP_K)
    plt.show()
else:
    print("No retrieval records available for visualization.")